# 02 PPO 微調（MaskablePPO + IL 熱啟動）

從 IL 訓練好的 encoder 出發，用課程式 reward 讓模型超越教師。

重點：`configs/ppo.yaml` 的 `train.il_warm_start` 指向 IL checkpoint；
`model.network` 必須與 IL 相同（本範例為 `resnet`）才吃得到全部權重。

In [ ]:
# Colab 重啟後 cwd 會回到 /content；這裡確保在專案目錄（缺少時自動從 GitHub clone）
import os
import subprocess

try:  # Colab 重啟後 Drive 會卸載，這裡重新掛載（非 Colab 環境會跳過）
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception as exc:
    print('（非 Colab 環境或已掛載，略過）', exc)

PROJECT = '/content/tetrio-ai'
REPO_URL = 'https://github.com/wallacechen0130/tetr_bot.git'
if not os.path.exists(os.path.join(PROJECT, 'requirements.txt')):
    subprocess.run(f'git clone --depth 1 {REPO_URL} {PROJECT}', shell=True, check=True)
os.chdir(PROJECT)

import importlib.util

# 缺套件才安裝（已裝過時直接跳過，重跑很快）
required = ['numpy', 'pandas', 'pyarrow', 'torch', 'stable_baselines3', 'sb3_contrib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print('缺少套件，安裝 requirements-colab.txt：', missing)
    subprocess.run('pip -q install -r requirements-colab.txt', shell=True, check=True)
else:
    print('依賴已齊全')
print('工作目錄:', os.getcwd())

In [ ]:
import os

os.environ.setdefault('TETRIO_AI_DRIVE', '/content/drive/MyDrive/tetrio-ai')
DRIVE_ROOT = os.environ['TETRIO_AI_DRIVE']
IL_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'il', 'best.pt')
PPO_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'ppo')
TB_DIR = os.path.join(DRIVE_ROOT, 'runs')
os.makedirs(PPO_CKPT, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)
print('IL:', IL_CKPT, os.path.exists(IL_CKPT))
print('PPO:', PPO_CKPT)

In [ ]:
# 短跑驗證（10 分鐘內可完成，確認 reward 有在上升）
!python -m scripts.train_ppo --timesteps 50000 --n-envs 4 --vec subproc \
    --il-weights {IL_CKPT} --out {PPO_CKPT}/ppo_smoke.json

In [ ]:
# 正式訓練（T4 約 1.5-3 小時；斷線可用 --resume 接續）
!python -m scripts.train_ppo --timesteps 2000000 --n-envs 8 --vec subproc \
    --il-weights {IL_CKPT} --out {PPO_CKPT}/ppo_full.json

In [ ]:
# Resume 範例
!python -m scripts.train_ppo --resume {PPO_CKPT}/ppo_500000_steps.zip --timesteps 1000000 --n-envs 8

In [ ]:
# 監控：TensorBoard（Colab 內嵌）
%load_ext tensorboard
%tensorboard --logdir {TB_DIR}

In [ ]:
# 評估：用同一份程式碼跑基準局（會輸出 JSON + Markdown）
!python -m scripts.evaluate --agent heuristic --env-id TetrisSurvival-v0 --episodes 3 \
    --out-json {PPO_CKPT}/report.json --out-md {PPO_CKPT}/report.md